# v23 - Adapting the public OAI CoAtNet-384 through the v07 adapter (E5, local GPU)

Design: `docs/research/v21-shortlist-design.md` revision 2, section 4 (decision D-020). A joint
preprocessing-and-weight adaptation: the public checkpoint is fed from our v07 cache through an
adapter instead of its own DICOM pipeline (Stage 0: Spearman 0.984 with the exact pipeline on gold),
then fine-tuned briefly on our target with a self-distillation term.

**Provenance.**

- Checkpoint: `raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt` (SHA-256 `7e5315dad125...`) from
  `nartaa/rsna-knee-publication-swa-weights-20261007`, trained by nartaa with 2,399 OAI knees (D-019).
  Terms (dataset README): "provided for research and educational use, including the RSNA Knee
  competition"; no rights to OAI data. Adapted weights stay private and are not published.
- Model code: the public `RaptorClassifier` (per-finding attention pooling over three-slice windows),
  `_eval_centers` and the anatomical mirror from `nartaa/rsna-knee-0949-anatomical-mirror` (Apache 2.0),
  rewritten here; backbone `coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k` from timm at 320 px, as there.
- Data: our v06 targets and folds, the v07 320 px cache, the v11 OOF teacher (target `round2`), and for
  target `T3` the CC0 Gemini table of `nartaa/rsna-knee-hpo-assets`.

**Adapter** (design 4.1): the five v07 slots map to the public slots (26/22/18/12/18 slices); a series
already used stays zero; public slice picks `linspace(int(0.02 n), int(0.98 n) - 1, k)` map to the
stored slice with the nearest original index; our canonical flips are undone; the centre 267 px of the
320 px slice (the public 116.7 mm field) is resized to 320 px; intensity 'restretch' (2-98 percentile of
the slot's stored pixels); K94 window centres and the mirror view as in the public code.

Modes (`MODE`): `stage0` (gold-58 through the adapter, compared with v18), `teacher` (the untouched
checkpoint on all cached studies, K94 + mirror; the self-distillation targets), `adapt` (design 4.2:
3 epochs on the 4,349 non-gold studies, loss `0.5 * weighted BCE(T*) + 0.5 * BCE(teacher)`, then the
admission report of design 4.3). Run locally with `scripts/local_v23.py` (D-016).

In [ ]:
# v23 configuration
import json
import os
import subprocess
import sys
import time
from pathlib import Path

VERSION = 'v23'
MODE = 'stage0'          # 'stage0', 'teacher' or 'adapt'
TARGET = 'round2'        # design 3 rule: 'T3' only if v22_submit >= 0.935 (v22 matched v13 F0; default kept)
INTENSITY = 'restretch'  # chosen in Stage 0 by fidelity to v18 (design 4.1)
NOTEBOOK_T0 = time.time()
WORK = Path('/kaggle/working') / VERSION
WORK.mkdir(parents=True, exist_ok=True)

LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA',
          'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
CKPT_NAME = 'raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt'
CKPT_SHA256 = '7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef'
OOF_WEIGHT = 0.5                 # T* = (1 - OOF_WEIGHT) * soft + OOF_WEIGHT * v11 OOF, as v13 / v22
OOF_FILES = {k: f'v11_oof_fold{k}_B_k16.csv' for k in range(5)}
GEMINI_FILE = 'labels_v1_gemlow.parquet'
ADAPT = {                        # design 4.2, fixed before the run
    'epochs': 3, 'lr_backbone': 1e-5, 'lr_head': 1e-4, 'weight_decay': 0.05, 'k_train': 16,
    'mirror_p': 0.5, 'accum': 8, 'teacher_weight': 0.5, 'grad_checkpointing': True, 'bn_eval': True,
    'clip_grad': 2.0, 'seed': 2026, 'drift_n': 300,
}
ADMISSION = {'min_gain': 0.003, 'max_p_nonpos': 0.2, 'max_finding_drop': 0.02, 'n_boot': 2000, 'seed': 0}
NUM_WORKERS = 4
print(f'{VERSION}: mode {MODE}, target {TARGET}, intensity {INTENSITY}', flush=True)

In [ ]:
# Locate the inputs without walking the competition image tree.
def find_unique_file(name, root=Path('/kaggle/input'), required=True):
    hits = []
    for current, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series', 'competitions', 'cache')
                   and 'rsna-knee-abnormality-detection' not in d]
        if name in files:
            hits.append(Path(current) / name)
    if len(hits) != 1:
        if not required and not hits:
            return None
        raise FileNotFoundError(f'expected one {name} under {root}, found {hits}')
    return hits[0]

PATHS = {
    'targets': str(find_unique_file('v06_targets.csv')),
    'folds': str(find_unique_file('v06_folds.csv')),
    'cache_studies': str(find_unique_file('v07_cache_studies.csv')),
    'cache_index': str(find_unique_file('v07_cache_index.csv')),
    'checkpoint': str(find_unique_file(CKPT_NAME)),
}
PATHS['cache_dir'] = str(Path(PATHS['cache_studies']).parent / 'cache')
assert Path(PATHS['cache_dir']).is_dir(), PATHS['cache_dir']
_v18 = find_unique_file('v18_gold_coatnet_ranks.csv', required=False)    # optional: Stage 0 / report comparison
PATHS['v18_gold'] = str(_v18) if _v18 else None
if MODE == 'adapt':
    OOF_PATHS = {k: find_unique_file(name) for k, name in OOF_FILES.items()}
    if TARGET == 'T3':
        PATHS['gemini'] = str(find_unique_file(GEMINI_FILE))
print(json.dumps(PATHS, indent=2))

## Training targets T* (adapt mode)

As v13 (`round2`) or v22 (`T3`): for every non-gold cached study `(1 - OOF_WEIGHT) * soft + OOF_WEIGHT *
v11 OOF`, with soft the v06 4-source target (`round2`) or `(n * soft4 + gemini) / (n + 1)` (`T3`). The
same checks as v13/v22. Written to `v23_train_targets.csv`.

In [ ]:
import hashlib
import numpy as np
import pandas as pd

if MODE == 'adapt':
    targets = pd.read_csv(PATHS['targets'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
    folds = pd.read_csv(PATHS['folds'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
    cached = pd.read_csv(PATHS['cache_studies'], dtype={'StudyInstanceUID': str})
    ok = set(cached[cached.error.fillna('') == ''].StudyInstanceUID)
    fold_of = folds.fold.reindex(targets.index)
    gold_ids = set(targets.index[targets.is_gold.astype(bool)])
    oof_parts = []
    for k, path in sorted(OOF_PATHS.items()):
        df = pd.read_csv(path, dtype={'StudyInstanceUID': str})
        assert df.columns.tolist() == ['StudyInstanceUID'] + LABELS and df.StudyInstanceUID.is_unique
        assert set(df.StudyInstanceUID) == {u for u in targets.index if fold_of[u] == k and u in ok}, path.name
        oof_parts.append(df.set_index('StudyInstanceUID'))
    oof = pd.concat(oof_parts)
    pool = [u for u in targets.index if fold_of[u] >= 0 and u in ok]
    assert set(pool) == set(oof.index) and not (set(oof.index) & gold_ids)
    soft = targets.loc[pool, LABELS].astype(np.float64)
    if TARGET == 'T3':
        gem = pd.read_parquet(PATHS['gemini']).set_index('StudyInstanceUID').loc[pool, LABELS].astype(np.float64)
        n_src = targets.loc[pool, [f'{c}__n_sources' for c in LABELS]].to_numpy(np.float64)
        soft = (n_src * soft + gem) / (n_src + 1.0)
    mixed = (1.0 - OOF_WEIGHT) * soft + OOF_WEIGHT * oof.loc[pool, LABELS].astype(np.float64)
    vals = mixed.to_numpy()
    assert np.isfinite(vals).all() and vals.min() >= 0 and vals.max() <= 1
    mixed.index.name = 'StudyInstanceUID'
    PATHS['train_targets'] = str(WORK / 'v23_train_targets.csv')
    mixed.to_csv(PATHS['train_targets'])
    PATHS['teacher'] = str(WORK / 'v23_teacher_probs.csv')
    assert Path(PATHS['teacher']).is_file(), 'run MODE teacher first'
    print(f'targets {TARGET}: {len(mixed)} studies; sha256 '
          f'{hashlib.sha256(Path(PATHS["train_targets"]).read_bytes()).hexdigest()[:16]}', flush=True)

## Core script

Written to disk and run in its own process (data-loader workers need importable classes). The notebook
is the source of truth; the script is regenerated on every run.

In [ ]:
%%writefile /kaggle/working/v23_core.py
"""v23 core: the public OAI CoAtNet-384 fed from the v07 cache (adapter of design v21-shortlist 4.1).
Modes: stage0 (gold through the adapter), teacher (all cached studies), adapt (design 4.2)."""
import hashlib
import json
import math
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset

LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA',
          'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
OUR_SLOTS = ['SAG_FS', 'SAG_NFS', 'COR_FS', 'COR_OTHER', 'AX_FS']
SLOT_SIZES = [26, 22, 18, 12, 18]          # public d96 layout, in the order of OUR_SLOTS
SAGITTAL_SLOTS = (0, 1)
D, RES, K_EVAL = 96, 320, 94
CROP_PX = int(round(320 * 320 / 384))      # 140 mm at 320 px -> the public 116.7 mm post-crop field (267 px)
SLOT_OF = np.searchsorted(np.cumsum([0] + SLOT_SIZES), np.arange(D), side='right') - 1
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

CFG = json.loads(sys.argv[1])
OUT = Path(CFG['out_dir'])
OUT.mkdir(parents=True, exist_ok=True)
LOG = OUT / f'v23_log_{CFG["mode"]}.txt'


def log(msg):
    line = f'[{time.strftime("%H:%M:%S")}] {msg}'
    print(line, flush=True)
    with open(LOG, 'a') as f:
        f.write(line + '\n')


# ---------------------------------------------------------------- model (public structure)
class RaptorClassifier(nn.Module):
    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))

    def forward(self, x):
        B, K = x.shape[:2]
        f = torch.cat([self.backbone(c) for c in x.flatten(0, 1).split(16)], 0).view(B, K, -1)
        h = self.norm(f)
        a = torch.softmax(self.att(h), dim=1)
        pooled = torch.einsum('bkn,bkf->bnf', a, h)
        return (pooled * self.clsW).sum(-1) + self.clsb


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


def load_public(path, expect_sha):
    assert sha256(path) == expect_sha, 'checkpoint bytes differ'
    ck = torch.load(path, map_location='cpu', weights_only=False)
    arch = ck['arch']
    bb = timm.create_model(arch, pretrained=False, num_classes=0, in_chans=3, img_size=RES, global_pool='avg')
    model = RaptorClassifier(bb, F_dim=bb.num_features)
    model.load_state_dict(ck['model'], strict=True)
    return model, arch


def set_train_mode(model, bn_eval):
    model.train()
    if bn_eval:
        for mod in model.backbone.modules():
            if isinstance(mod, nn.modules.batchnorm._BatchNorm):
                mod.eval()


# ------------------------------------------------------------------------- adapter
def slot_rows(index_csv):
    idx = pd.read_csv(index_csv, dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
    out = {}
    for r in idx.itertuples(index=False):
        out.setdefault(r.StudyInstanceUID, {})[r.slot] = {
            'present': str(r.present) == 'True', 'series': r.SeriesInstanceUID, 'n': int(r.n_series_slices) if str(r.present) == 'True' else 0,
            'flip_lr': str(r.flip_lr) == 'True', 'flip_ud': str(r.flip_ud) == 'True'}
    return out


def build_volume(uid, rows, cache_dir, intensity):
    with np.load(Path(cache_dir) / f'{uid}.npz') as z:
        jpeg, offsets, slot, raw = z['jpeg'], z['offsets'], z['slot'], z['raw_index']
    vol = np.zeros((D, RES, RES), np.uint8)
    used, pos = set(), 0
    for s, (name, k) in enumerate(zip(OUR_SLOTS, SLOT_SIZES)):
        r = rows.get(name)
        sel = np.flatnonzero(slot == s)
        if r is None or not r['present'] or len(sel) == 0 or r['series'] in used:
            pos += k                                       # public rule: each series once; an empty slot stays zero
            continue
        used.add(r['series'])
        n = r['n']
        lo, hi = int(n * 0.02), int(n * 0.98) - 1
        hi = max(hi, lo)
        picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else np.zeros(k, int)
        raws = raw[sel].astype(int)
        imgs = []
        for p in picks:
            j = sel[int(np.argmin(np.abs(raws - p)))]
            a = cv2.imdecode(np.frombuffer(jpeg[offsets[j]:offsets[j + 1]].tobytes(), np.uint8), cv2.IMREAD_GRAYSCALE)
            if r['flip_lr']:
                a = a[:, ::-1]                             # undo the v07 canonical flips (DICOM pixel orientation)
            if r['flip_ud']:
                a = a[::-1, :]
            imgs.append(a)
        imgs = np.stack(imgs).astype(np.float32)
        if intensity == 'restretch':
            q2, q98 = np.percentile(imgs, [2.0, 98.0])
            imgs = np.clip((imgs - q2) / (q98 - q2 + 1e-6), 0, 1) * 255.0
        c0 = (RES - CROP_PX) // 2
        for a in imgs:
            a = np.ascontiguousarray(a[c0:c0 + CROP_PX, c0:c0 + CROP_PX])
            vol[pos] = np.clip(cv2.resize(a, (RES, RES), interpolation=cv2.INTER_LINEAR), 0, 255).round().astype(np.uint8)
            pos += 1
    mask = (vol.reshape(D, -1).sum(1) > 0).astype(np.uint8)
    return vol, mask


def valid_centres(mask):
    valid = np.where(mask > 0)[0]
    if len(valid) < 3:
        valid = np.arange(min(3, D))
    lo, hi = int(valid.min()), int(valid.max())
    return [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi] or [max(1, min((lo + hi) // 2, D - 2))]


def eval_centres(mask, k=K_EVAL):
    cs = valid_centres(mask)
    return [cs[i] for i in np.linspace(0, len(cs) - 1, k).round().astype(int)]


def train_centres(mask, k, rng):
    cs = valid_centres(mask)
    idx = rng.choice(len(cs), size=k, replace=len(cs) < k)
    return sorted(cs[i] for i in idx)


def windows(vol_t, centres, mirror):
    # vol_t: (D, H, W) uint8 on the device. Public order: mirror on uint8, then /255 and ImageNet normalisation.
    cs = np.clip(np.asarray(centres), 1, D - 2)
    tri = vol_t[torch.as_tensor(np.stack([cs - 1, cs, cs + 1], 1), device=vol_t.device)]   # (K, 3, H, W)
    if mirror:
        sag = torch.as_tensor(np.isin(SLOT_OF[cs], SAGITTAL_SLOTS), device=vol_t.device).view(-1, 1, 1, 1)
        tri = torch.where(sag, tri.flip(1), tri.flip(-1))
    x = tri.float() / 255.0
    return (x - MEAN.to(x.device)) / STD.to(x.device)


class VolumeDS(Dataset):
    def __init__(self, uids, rows, cache_dir, intensity, y=None, w=None, p=None, train=False, seed=0):
        self.uids, self.rows, self.cache_dir, self.intensity = list(uids), rows, cache_dir, intensity
        self.y, self.w, self.p, self.train, self.seed, self.epoch = y, w, p, train, seed, 0

    def __len__(self):
        return len(self.uids)

    def __getitem__(self, i):
        uid = self.uids[i]
        vol, mask = build_volume(uid, self.rows[uid], self.cache_dir, self.intensity)
        item = {'uid': uid, 'vol': vol, 'mask': mask}
        if self.train:
            rng = np.random.default_rng((self.seed, self.epoch, i))
            item.update(centres=train_centres(mask, CFG['k_train'], rng), mirror=bool(rng.random() < CFG['mirror_p']),
                        y=self.y[i], w=self.w[i], p=self.p[i])
        return item


def first(batch):
    return batch[0]


@torch.no_grad()
def predict(model, uids, rows, device):
    # Untouched-pipeline inference: K94 windows, mean of plain and mirrored probabilities, fp16.
    model.eval()
    ds = VolumeDS(uids, rows, CFG['cache_dir'], CFG['intensity'])
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=CFG['num_workers'], collate_fn=first)
    out, t0 = {}, time.time()
    for n, item in enumerate(loader):
        vol = torch.from_numpy(item['vol']).to(device)
        cs = eval_centres(item['mask'])
        ps = []
        for mirror in (False, True):
            with torch.autocast('cuda', dtype=torch.float16):
                z = model(windows(vol, cs, mirror).unsqueeze(0)).float()
            assert torch.isfinite(z).all()
            ps.append(torch.sigmoid(z)[0].cpu().numpy())
        out[item['uid']] = 0.5 * (ps[0] + ps[1])
        if (n + 1) % 200 == 0:
            log(f'predicted {n + 1}/{len(uids)} studies, {(n + 1) / (time.time() - t0):.2f} studies/s')
    return pd.DataFrame.from_dict(out, orient='index', columns=LABELS).loc[list(uids)]


def macro_auc(y, p):
    per = {c: float(roc_auc_score(y[:, j], p[:, j])) for j, c in enumerate(LABELS) if 0 < y[:, j].sum() < len(y)}
    return float(np.mean(list(per.values()))), per


# ---------------------------------------------------------------------------- main
def main():
    mode, device = CFG['mode'], torch.device('cuda')
    torch.backends.cudnn.benchmark = True
    t_start = time.time()
    model, arch = load_public(CFG['checkpoint'], CFG['checkpoint_sha256'])
    model.to(device)
    rows = slot_rows(CFG['cache_index'])
    targets = pd.read_csv(CFG['targets'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
    cached = pd.read_csv(CFG['cache_studies'], dtype={'StudyInstanceUID': str})
    ok = set(cached[cached.error.fillna('') == ''].StudyInstanceUID)
    gold_ids = [u for u in targets.index if bool(targets.loc[u, 'is_gold']) and u in ok]
    y_gold = targets.loc[gold_ids, [f'{c}__gold' for c in LABELS]].to_numpy(np.float32)
    receipt = {'version': 'v23', 'mode': mode, 'arch': arch, 'checkpoint_sha256': CFG['checkpoint_sha256'],
               'intensity': CFG['intensity'], 'torch': torch.__version__, 'timm': timm.__version__,
               'device_name': torch.cuda.get_device_name(0)}
    log(f'mode {mode}: {len(ok)} cached studies, {len(gold_ids)} gold, arch {arch}')

    def gold_report(p, tag):
        m, per = macro_auc(y_gold, p.loc[gold_ids].to_numpy())
        rec = {'gold_macro': m, 'gold_per_label': per}
        if CFG.get('v18_gold'):
            v18 = pd.read_csv(CFG['v18_gold'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID').loc[gold_ids, LABELS]
            rho = {c: float(spearmanr(p.loc[gold_ids, c], v18[c]).statistic) for c in LABELS}
            rec.update(spearman_vs_v18=rho, spearman_vs_v18_mean=float(np.mean(list(rho.values()))),
                       spearman_vs_v18_min=float(min(rho.values())), v18_gold_macro=macro_auc(y_gold, v18.to_numpy())[0])
        log(f'{tag}: gold {m:.4f}' + (f', Spearman vs v18 mean {rec["spearman_vs_v18_mean"]:.3f} min '
                                      f'{rec["spearman_vs_v18_min"]:.3f}' if 'spearman_vs_v18_mean' in rec else ''))
        return rec

    if mode == 'stage0':
        p = predict(model, gold_ids, rows, device)
        p.index.name = 'StudyInstanceUID'
        p.to_csv(OUT / 'v23_gold_untouched.csv')
        receipt['untouched'] = gold_report(p, 'untouched')
    elif mode == 'teacher':
        uids = [u for u in targets.index if u in ok]
        p = predict(model, uids, rows, device)
        p.index.name = 'StudyInstanceUID'
        p.to_csv(OUT / 'v23_teacher_probs.csv')
        receipt.update(n_studies=len(uids), untouched=gold_report(p, 'untouched (teacher run)'))
    elif mode == 'adapt':
        fit = pd.read_csv(CFG['train_targets'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
        teacher = pd.read_csv(CFG['teacher'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
        train_ids = [u for u in fit.index if u in ok]
        assert not set(train_ids) & set(gold_ids)
        Y = fit.loc[train_ids, LABELS].to_numpy(np.float32)
        W = targets.loc[train_ids, [f'{c}__weight' for c in LABELS]].to_numpy(np.float32)
        P = teacher.loc[train_ids, LABELS].to_numpy(np.float32)
        if CFG['grad_checkpointing']:
            model.backbone.set_grad_checkpointing(True)
        head = [p for n, p in model.named_parameters() if not n.startswith('backbone.')]
        opt = torch.optim.AdamW([{'params': list(model.backbone.parameters()), 'lr': CFG['lr_backbone']},
                                 {'params': head, 'lr': CFG['lr_head']}], weight_decay=CFG['weight_decay'])
        base_lrs = [CFG['lr_backbone'], CFG['lr_head']]
        accum = CFG['accum']
        total_steps = CFG['epochs'] * math.ceil(len(train_ids) / accum)
        scaler = torch.amp.GradScaler('cuda')
        ds = VolumeDS(train_ids, rows, CFG['cache_dir'], CFG['intensity'], Y, W, P, train=True, seed=CFG['seed'])
        tw = CFG['teacher_weight']
        step, micro, n_img, t_train, history = 0, 0, 0, time.time(), []
        torch.cuda.reset_peak_memory_stats()
        for epoch in range(CFG['epochs']):
            ds.epoch = epoch
            loader = DataLoader(ds, batch_size=1, shuffle=True, num_workers=CFG['num_workers'], collate_fn=first,
                                generator=torch.Generator().manual_seed(CFG['seed'] + epoch))
            set_train_mode(model, CFG['bn_eval'])
            opt.zero_grad(set_to_none=True)
            sums = np.zeros(3)
            for item in loader:
                vol = torch.from_numpy(item['vol']).to(device, non_blocking=True)
                x = windows(vol, item['centres'], item['mirror']).unsqueeze(0)
                y = torch.from_numpy(item['y'][None]).to(device)
                w = torch.from_numpy(item['w'][None]).to(device)
                p = torch.from_numpy(item['p'][None]).to(device)
                with torch.autocast('cuda', dtype=torch.float16):
                    z = model(x).float()
                l_target = (F.binary_cross_entropy_with_logits(z, y, reduction='none') * w).sum() / w.sum()
                l_teacher = F.binary_cross_entropy_with_logits(z, p)
                loss = (1 - tw) * l_target + tw * l_teacher
                scaler.scale(loss / accum).backward()
                micro += 1
                n_img += x.shape[1]
                sums += (loss.item(), l_target.item(), l_teacher.item())
                if micro % accum == 0:
                    lr_f = 0.5 * (1 + math.cos(math.pi * min(step, total_steps) / total_steps))
                    for g, b in zip(opt.param_groups, base_lrs):
                        g['lr'] = b * lr_f
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['clip_grad'])
                    scaler.step(opt)
                    scaler.update()
                    opt.zero_grad(set_to_none=True)
                    step += 1
                if micro % 200 == 0:
                    log(f'epoch {epoch} micro {micro} loss {sums[0] / 200:.4f} (target {sums[1] / 200:.4f}, '
                        f'teacher {sums[2] / 200:.4f}) img/s {n_img / (time.time() - t_train):.1f}')
                    sums[:] = 0
            p_gold = predict(model, gold_ids, rows, device)
            rec = gold_report(p_gold, f'epoch {epoch}')
            history.append({'epoch': epoch, 'gold_macro': rec['gold_macro'], 'steps': step,
                            'elapsed_s': round(time.time() - t_start, 1)})
        cfg_contract = {'version': 'v23', 'arch': arch, 'timm_kwargs': {'img_size': RES, 'global_pool': 'avg', 'in_chans': 3},
                        'head': 'RaptorClassifier', 'f_dim': model.backbone.num_features, 'adapter': {
                            'slots': OUR_SLOTS, 'slot_sizes': SLOT_SIZES, 'crop_px': CROP_PX, 'res': RES,
                            'intensity': CFG['intensity'], 'k_eval': K_EVAL, 'mirror_tta': True},
                        'base_checkpoint_sha256': CFG['checkpoint_sha256'], 'target': CFG['target'],
                        **{k: CFG[k] for k in ('epochs', 'lr_backbone', 'lr_head', 'weight_decay', 'k_train', 'mirror_p',
                                               'accum', 'teacher_weight', 'bn_eval', 'seed')}}
        ckpt = OUT / 'v23_adapt_last.pt'
        torch.save({'model': model.state_dict(), 'cfg': cfg_contract}, ckpt)
        p_gold.index.name = 'StudyInstanceUID'
        p_gold.to_csv(OUT / 'v23_gold_adapted.csv')
        rng = np.random.default_rng(CFG['seed'])
        drift_ids = sorted(rng.choice(train_ids, size=min(CFG['drift_n'], len(train_ids)), replace=False).tolist())
        p_drift = predict(model, drift_ids, rows, device)
        drift = {c: {'mean_abs_diff': float(np.abs(p_drift[c] - teacher.loc[drift_ids, c]).mean()),
                     'spearman': float(spearmanr(p_drift[c], teacher.loc[drift_ids, c]).statistic)} for c in LABELS}
        receipt.update(n_train=len(train_ids), optimizer_steps=step, micro_steps=micro, train_images=n_img,
                       train_images_per_s=round(n_img / (time.time() - t_train), 1), history=history,
                       peak_allocated_gb=round(torch.cuda.max_memory_allocated() / 2**30, 2),
                       adapted=gold_report(p_gold, 'adapted (last epoch)'), drift_train_vs_teacher=drift,
                       checkpoint=str(ckpt), checkpoint_sha256=sha256(ckpt), cfg_contract=cfg_contract)
    receipt['status'] = 'done'
    receipt['total_seconds'] = round(time.time() - t_start, 1)
    (OUT / f'v23_receipt_{mode}.json').write_text(json.dumps(receipt, indent=2, default=float))
    log(f'done in {receipt["total_seconds"]} s')


if __name__ == '__main__':
    main()

## Run the mode

One process on one GPU; its log goes to `v23_log_<mode>.txt` and its receipt to `v23_receipt_<mode>.json`.

In [ ]:
cfg = {**ADAPT, **PATHS, 'mode': MODE, 'target': TARGET, 'intensity': INTENSITY, 'checkpoint_sha256': CKPT_SHA256,
       'num_workers': NUM_WORKERS, 'out_dir': str(WORK)}
receipt_file = WORK / f'v23_receipt_{MODE}.json'
receipt_file.unlink(missing_ok=True)
proc = subprocess.run([sys.executable, '/kaggle/working/v23_core.py', json.dumps(cfg)])
receipt = json.loads(receipt_file.read_text()) if receipt_file.is_file() else {'status': 'missing'}
print(f'{MODE}: exit code {proc.returncode}, status {receipt.get("status")}, '
      f'{(time.time() - NOTEBOOK_T0) / 3600:.2f} h', flush=True)
assert proc.returncode == 0 and receipt.get('status') == 'done', 'the run did not complete'

## Admission report (adapt mode, design 4.3)

Gold-58 for three outputs: v18 (the exact public pipeline), the untouched checkpoint through the adapter
(teacher run), the adapted model through the adapter (last epoch). Submission only if the adapted model
beats the untouched adapter by >= +0.003 with P(gain <= 0) <= 0.2 (paired bootstrap over studies), beats
v18 exact, and loses no finding by more than 0.02 against the untouched adapter.

In [ ]:
if MODE == 'adapt':
    from sklearn.metrics import roc_auc_score
    tg = pd.read_csv(PATHS['targets'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
    g_ids = pd.read_csv(WORK / 'v23_gold_adapted.csv', dtype={'StudyInstanceUID': str}).StudyInstanceUID.tolist()
    yg = tg.loc[g_ids, [f'{c}__gold' for c in LABELS]].to_numpy(int)
    outs = {'adapted': pd.read_csv(WORK / 'v23_gold_adapted.csv', dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID'),
            'untouched': pd.read_csv(PATHS['teacher'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')}
    if PATHS['v18_gold']:
        outs['v18_exact'] = pd.read_csv(PATHS['v18_gold'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
    arr = {k: v.loc[g_ids, LABELS].to_numpy(float) for k, v in outs.items()}

    def macro(y, p):
        return float(np.mean([roc_auc_score(y[:, j], p[:, j]) for j in range(len(LABELS))]))

    def paired(a, b):
        rng = np.random.default_rng(ADMISSION['seed'])
        d = []
        for _ in range(ADMISSION['n_boot']):
            s = rng.integers(0, len(yg), len(yg))
            if all(0 < yg[s, j].sum() < len(s) for j in range(len(LABELS))):
                d.append(macro(yg[s], arr[b][s]) - macro(yg[s], arr[a][s]))
        d = np.asarray(d)
        return {'gain': float(d.mean()), 'ci': [float(np.quantile(d, 0.025)), float(np.quantile(d, 0.975))],
                'p_nonpos': float((d <= 0).mean())}

    per = pd.DataFrame({k: [roc_auc_score(yg[:, j], v[:, j]) for j in range(len(LABELS))] for k, v in arr.items()}, index=LABELS)
    report = {'gold_macro': {k: macro(yg, v) for k, v in arr.items()}, 'adapted_vs_untouched': paired('untouched', 'adapted')}
    if 'v18_exact' in arr:
        report['adapted_vs_v18_exact'] = paired('v18_exact', 'adapted')
    drop = (per['untouched'] - per['adapted']).max()
    g = report['adapted_vs_untouched']
    report['admitted'] = bool(g['gain'] >= ADMISSION['min_gain'] and g['p_nonpos'] <= ADMISSION['max_p_nonpos']
                              and ('v18_exact' not in arr or report['gold_macro']['adapted'] > report['gold_macro']['v18_exact'])
                              and drop <= ADMISSION['max_finding_drop'])
    report['max_finding_drop_vs_untouched'] = float(drop)
    report['per_finding'] = per.round(4).to_dict()
    (WORK / 'v23_admission.json').write_text(json.dumps(report, indent=2))
    print(per.round(3).to_string())
    print(json.dumps({k: v for k, v in report.items() if k != 'per_finding'}, indent=2))